# Qwen2.5-1.5B + LoRA: `[REPLY]` generator / `[TRACK]` tracker

Fine-tunes one of the two LoRA adapters `backend/src/llm.py` serves on the same base model, then
evaluates it and pushes it (adapter + tokenizer) to the Hub. Owners: Denise (`[REPLY]`), Nichole (`[TRACK]`).

- **GPU cluster / any Jupyter:** start Jupyter from `training/` in a checkout of the repo (the notebook
  uses that checkout; outputs go to the gitignored `outputs/`), install `backend/requirements-ml.txt`
  plus `bitsandbytes datasets`, and `export HF_TOKEN=...` (write access) before starting it. Headless:
  `jupyter nbconvert --to notebook --execute qwen_denise.ipynb`.
- **Colab:** GPU runtime (T4 is enough), with secrets `HF_TOKEN` (write access, to push) and
  `GITHUB_TOKEN` (read access, to clone the repo). Checkpoints go to Drive.
- **`TASK = "reply"`**: the generator's adapter, `[REPLY]` rows only (minutes on a T4). Pushed to
  `makan-qwen-reply`, served as `MAKAN_GENERATOR_ADAPTER`.
- **`TASK = "track"`**: the tracker's adapter, `[TRACK]` rows only (a few hours on a T4). Pushed to
  `makan-qwen-track`, served as `MAKAN_TRACKER_ADAPTER`.
- **`TASK = "mix"`**: one adapter on both (`[REPLY]` upsampled), the shared-adapter ablation. Pushed to
  `makan-qwen-shared`; point both `MAKAN_*_ADAPTER` variables at it to compare with the separate ones.

Training sees exactly the text serving sees: the prompt goes in as one user message through the chat
template (`llm.complete()`), and the loss is computed on the answer only.

In [ ]:
TASK = "reply"            # "reply": generator [REPLY] | "track": tracker [TRACK] | "mix": both (ablation)
REPLY_UPSAMPLE = 3        # "mix" only: each [REPLY] row is repeated this many times (it is ~10% of the data)
# [REPLY] buckets (action kind, style) repeated this many times in training, on top of REPLY_UPSAMPLE:
# the ones the first reply adapter failed most (style_mismatch, unexplained_fallback). Others: 1.
BUCKET_UPSAMPLE = {("recommend", "singlish"): 3, ("fallback_recommend", "english"): 3,
                   ("fallback_recommend", "singlish"): 2}
LIMIT = None              # use only this many training rows (smoke test); None = all

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"   # must match MAKAN_BASE_LLM in the backend
HUB_REPO = {"reply": "makan-where-cs425/makan-qwen-reply", "track": "makan-where-cs425/makan-qwen-track",
            "mix": "makan-where-cs425/makan-qwen-shared"}[TASK]   # private; pushed only by the last cell
USE_4BIT = True           # QLoRA. False = plain fp16 LoRA: also fits a T4 and trains faster
MAX_LEN = 1024            # longer rows (a few long MultiWOZ dialogues) are dropped, not truncated

EPOCHS = 3 if TASK == "reply" else 2   # the reply data is small; the best checkpoint by dev loss is kept
LR = 2e-4
BATCH, GRAD_ACCUM = 4, 4  # effective batch 16
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
EVAL_ROWS = 400           # dev rows for the loss during training
EVAL_N = 200              # dev rows generated per task / source in the eval cells
SEED = 42

REPO_URL = "github.com/deniseLie/CS425-Makan-Where.git"

In [ ]:
%pip install -q -U peft accelerate bitsandbytes datasets

## Setup: repo, secrets, output folder

On Colab this mounts Drive (checkpoints survive a disconnect, and training resumes from the last
one), reads the secrets and clones the repo. Run locally from `training/`, it uses your checkout.

In [ ]:
import os, subprocess, sys
from pathlib import Path

try:
    from google.colab import drive, userdata
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

if ON_COLAB:
    drive.mount("/content/drive")
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN").strip()
    REPO_DIR = Path("/content/CS425-Makan-Where")
    OUT_DIR = Path(f"/content/drive/MyDrive/makan-qwen/{TASK}")
    if not REPO_DIR.exists():
        gh = userdata.get("GITHUB_TOKEN").strip()
        # no check=True: its error message would print the URL, token included
        res = subprocess.run(["git", "clone", "-q", "--depth", "1", f"https://x-access-token:{gh}@{REPO_URL}",
                              str(REPO_DIR)], capture_output=True, text=True)
        if res.returncode:
            raise RuntimeError("git clone failed: " + (res.stderr.replace(gh, "***").strip() or "no message"))
        subprocess.run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", f"https://{REPO_URL}"])
else:
    REPO_DIR = Path.cwd().parent              # run from training/ in a local checkout
    OUT_DIR = REPO_DIR / "outputs" / f"qwen_{TASK}"   # outputs/ is gitignored

assert (REPO_DIR / "backend" / "src").exists(), REPO_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(REPO_DIR / "backend"))
print("repo:", REPO_DIR, "| out:", OUT_DIR)

## Data

The rows are already in the serving format: `input` was built with `tracker.hf.build_prompt` /
`generator.hf.build_prompt`, and `output` is the gold answer. Rebuild the data files (not this
notebook) if those prompt functions change.

In [ ]:
import json, random
from collections import Counter

D = REPO_DIR / "data"
FILES = {
    "train": {"reply": ["reply_data/reply_train.jsonl"],
              "track": ["woz_converted_data/dialogues_train.jsonl", "synthetic_data/dialogues_synth_train.jsonl"]},
    "dev": {"reply": ["reply_data/reply_dev.jsonl"],
            "track": ["woz_converted_data/dialogues_dev.jsonl", "synthetic_data/dialogues_synth_dev.jsonl"]},
}

def read(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def load(split):
    return {task: [r for f in files for r in read(D / f)] for task, files in FILES[split].items()}

train_rows, dev_rows = load("train"), load("dev")
for split, rows in (("train", train_rows), ("dev", dev_rows)):
    print(split, {task: Counter(r["source"] for r in rs) for task, rs in rows.items()})

def bucket(row):  # (action kind, style) of a [REPLY] row, read from its prompt's Action line
    action = json.loads(row["input"].split("\nAction: ", 1)[1].split("\n", 1)[0])
    return action["kind"], row["style"]

reply_train = [r for r in train_rows["reply"] for _ in range(BUCKET_UPSAMPLE.get(bucket(r), 1))]
print("[REPLY] train rows per bucket, after BUCKET_UPSAMPLE:", dict(sorted(Counter(map(bucket, reply_train)).items())))

rng = random.Random(SEED)
if TASK == "reply":
    train_mix, dev_mix = reply_train, dev_rows["reply"]
elif TASK == "track":
    train_mix, dev_mix = train_rows["track"], dev_rows["track"]
elif TASK == "mix":
    train_mix = train_rows["track"] + reply_train * REPLY_UPSAMPLE
    dev_mix = dev_rows["track"] + dev_rows["reply"]
else:
    raise ValueError(TASK)
rng.shuffle(train_mix)
train_mix = train_mix[:LIMIT]
dev_mix = rng.sample(dev_mix, min(EVAL_ROWS, len(dev_mix)))
print(f"{TASK}: {len(train_mix)} train rows", Counter(r["task"] for r in train_mix), f"| {len(dev_mix)} dev rows")

### Tokenize

`chat()` is the text `llm.complete()` gives the model: the prompt as the only user message (Qwen's
template adds its default system message), then the assistant turn. Labels are `-100` on the prompt, so
only the answer and its closing `<|im_end|>` are learned.

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(BASE_MODEL)

def chat(prompt, answer=None):
    msgs = [{"role": "user", "content": prompt}]
    if answer is None:
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return tok.apply_chat_template(msgs + [{"role": "assistant", "content": answer}], tokenize=False)

def encode(row):
    prompt = tok(chat(row["input"]), add_special_tokens=False)["input_ids"]
    full = tok(chat(row["input"], row["output"]), add_special_tokens=False)["input_ids"]
    assert full[:len(prompt)] == prompt, row.get("id") or row.get("dialogue_id")
    return {"input_ids": full, "attention_mask": [1] * len(full),
            "labels": [-100] * len(prompt) + full[len(prompt):]}

def encode_all(rows):
    enc = [encode(r) for r in rows]
    kept = [e for e in enc if len(e["input_ids"]) <= MAX_LEN]
    lens = sorted(len(e["input_ids"]) for e in enc)
    print(f"{len(rows)} rows, tokens median {lens[len(lens) // 2]} / max {lens[-1]}; "
          f"dropped {len(enc) - len(kept)} over {MAX_LEN}")
    return kept

train_enc, dev_enc = encode_all(train_mix), encode_all(dev_mix)

ex = train_enc[0]  # the trained part of one example, after the prompt
print(tok.decode([t for t, l in zip(ex["input_ids"], ex["labels"]) if l != -100]))

## Model

In [ ]:
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

GPU = torch.cuda.is_available()
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                           bnb_4bit_compute_dtype=torch.float16) if USE_4BIT else None
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=quant, device_map="auto",
                                             torch_dtype=torch.float16 if GPU else torch.float32)
if USE_4BIT:
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
else:
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()

lora = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
model = get_peft_model(model, lora)
model.print_trainable_parameters()

## Train

Evaluates and checkpoints about 10 times per run. Re-running this cell after a disconnect resumes
from the newest checkpoint in `OUT_DIR`.

In [ ]:
import math
from datasets import Dataset
from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments

tok.padding_side = "right"
steps = math.ceil(len(train_enc) / (BATCH * GRAD_ACCUM)) * EPOCHS
every = max(10, steps // 10)
args = TrainingArguments(
    output_dir=str(OUT_DIR), num_train_epochs=EPOCHS, learning_rate=LR, lr_scheduler_type="cosine",
    warmup_steps=max(1, int(0.03 * steps)), per_device_train_batch_size=BATCH, per_device_eval_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM, gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False}, fp16=GPU,
    optim="paged_adamw_8bit" if USE_4BIT else "adamw_torch",
    logging_steps=max(1, every // 5), eval_strategy="steps", eval_steps=every, save_strategy="steps",
    save_steps=every, save_total_limit=2, load_best_model_at_end=True, metric_for_best_model="eval_loss",
    report_to="none", seed=SEED,
)
trainer = Trainer(model=model, args=args, train_dataset=Dataset.from_list(train_enc),
                  eval_dataset=Dataset.from_list(dev_enc),
                  data_collator=DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100))
print(f"{steps} optimizer steps, eval + checkpoint every {every}")
trainer.train(resume_from_checkpoint=any(OUT_DIR.glob("checkpoint-*")) or None)

FINAL = OUT_DIR / "final"
model.save_pretrained(FINAL)
tok.save_pretrained(FINAL)
print("saved", FINAL)

## Evaluate

Generation matches `llm.complete()`: greedy, up to 256 new tokens, the model's own generation config.
Predictions are written to `OUT_DIR` for reading through.

### `[REPLY]`: automatic reply checks (`TASK = "reply"` or `"mix"`)

Each dev reply goes through `generator.checks.check_reply`, the same filter every teacher reply passed
(100%): place names only from the action, every ask option named, no invented prices or distances, the
fallback explained, the right style. The catalogue is every place name in the reply data (the made-up
places the actions use).

In [ ]:
from src.generator.checks import check_reply
from src.schemas import PolicyAction

@torch.no_grad()
def generate(prompts, batch=16, max_new_tokens=256):
    model.eval()
    tok.padding_side = "left"
    outs = []
    for i in range(0, len(prompts), batch):
        enc = tok([chat(p) for p in prompts[i:i + batch]], return_tensors="pt", padding=True,
                  add_special_tokens=False).to(model.device)
        gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True,
                             pad_token_id=tok.pad_token_id or tok.eos_token_id)
        outs += tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return [o.strip() for o in outs]

def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        f.writelines(json.dumps(r, ensure_ascii=False) + "\n" for r in rows)

if TASK in ("reply", "mix"):
    full = {r["id"]: r for r in read(D / "reply_data/replies_dev.jsonl")}  # action, style, user_text per id
    catalogue = sorted({p["name"] for f in ("replies_train.jsonl", "replies_dev.jsonl")
                        for r in read(D / "reply_data" / f) for p in r["action"]["places"]})
    rows = random.Random(SEED).sample(dev_rows["reply"], min(EVAL_N, len(dev_rows["reply"])))
    preds = generate([r["input"] for r in rows])

    results, codes, passed = [], Counter(), Counter()
    for r, pred in zip(rows, preds):
        src = full[r["id"]]
        action = PolicyAction.from_dict(src["action"])
        failures = check_reply(pred, action, src["style"], src["user_text"], catalogue)
        codes.update(c for c, _ in failures)
        passed[(action.kind, src["style"], not failures)] += 1
        results.append({"id": r["id"], "kind": action.kind, "style": src["style"], "user_text": src["user_text"],
                        "pred": pred, "gold": r["output"], "failures": failures})
    write_jsonl(OUT_DIR / "eval_reply.jsonl", results)

    n_ok = sum(not r["failures"] for r in results)
    print(f"[REPLY] pass rate {n_ok}/{len(results)} = {n_ok / len(results):.1%}")
    for kind in ("ask", "recommend", "fallback_recommend"):
        for style in ("singlish", "english"):
            ok, bad = passed[(kind, style, True)], passed[(kind, style, False)]
            if ok + bad:
                print(f"  {kind:19s} {style:9s} {ok}/{ok + bad}")
    print("failures:", dict(codes.most_common()))
    for r in results[:8]:
        print(f"\n[{r['kind']}, {r['style']}] user: {r['user_text']}\n  pred: {r['pred']}\n  gold: {r['gold']}"
              + (f"\n  FAILED: {r['failures']}" if r["failures"] else ""))

### `[TRACK]`: slot accuracy (`TASK = "track"` or `"mix"`)

Per source (MultiWOZ, synthetic Singlish): joint goal accuracy (all six slots right), slot accuracy,
F1 over the slots the user actually filled (gold not `UNKNOWN`), feedback accuracy, and how often the
output was not valid JSON. Values are compared case-insensitively. For the final numbers, run the
MultiWOZ test file (`dialogues_test.jsonl`) once at the end.

In [ ]:
from src.schemas import SLOTS

def parse_json(text):
    try:
        d = json.loads(text[text.index("{"):text.rindex("}") + 1])
        return d if isinstance(d, dict) else None
    except ValueError:
        return None

def norm(v):
    return str(v).strip().lower()

def track_scores(rows, preds):
    n = len(rows)
    joint = slot_ok = fb_ok = invalid = tp = fp = fn = 0
    for r, pred in zip(rows, preds):
        gold, got = json.loads(r["output"]), parse_json(pred)
        if got is None:
            invalid += 1
            got = {}
        right = [norm(got.get(s, "UNKNOWN")) == norm(gold[s]) for s in SLOTS]
        joint += all(right)
        slot_ok += sum(right)
        fb_ok += norm(got.get("feedback", "none")) == norm(gold["feedback"])
        for s, ok in zip(SLOTS, right):
            g, p = norm(gold[s]), norm(got.get(s, "UNKNOWN"))
            tp += ok and g != "unknown"
            fp += not ok and p != "unknown"
            fn += not ok and g != "unknown"
    f1 = 2 * tp / (2 * tp + fp + fn) if tp else 0.0
    return {"n": n, "joint_acc": joint / n, "slot_acc": slot_ok / (n * len(SLOTS)), "slot_f1": f1,
            "feedback_acc": fb_ok / n, "invalid_json": invalid / n}

if TASK in ("track", "mix"):
    for source in ("multiwoz", "synthetic"):
        pool = [r for r in dev_rows["track"] if r["source"] == source]
        rows = random.Random(SEED).sample(pool, min(EVAL_N, len(pool)))
        preds = generate([r["input"] for r in rows], batch=8)
        print(source, {k: round(v, 3) for k, v in track_scores(rows, preds).items()})
        write_jsonl(OUT_DIR / f"eval_track_{source}.jsonl",
                    [{"input": r["input"], "pred": p, "gold": r["output"]} for r, p in zip(rows, preds)])

## Push to the Hub

Only after the eval looks good. The best checkpoint by dev loss is what `trainer` loaded at the end.
The backend serves `makan-qwen-reply` (TASK = "reply") and `makan-qwen-track` (TASK = "track") by
default; its `.env` (or the environment) only needs:

```bash
HF_TOKEN=hf_...                 # a read token from a makan-where-cs425 member
# TASK = "mix" (ablation): point both at the shared adapter
# MAKAN_TRACKER_ADAPTER=makan-where-cs425/makan-qwen-shared
# MAKAN_GENERATOR_ADAPTER=makan-where-cs425/makan-qwen-shared
```

In [ ]:
model.push_to_hub(HUB_REPO, private=True, commit_message=f"Qwen LoRA, TASK={TASK}, {EPOCHS} epoch(s)")
tok.push_to_hub(HUB_REPO, private=True)
print(f"pushed https://huggingface.co/{HUB_REPO}")